# Find and download CoRE Stack GeoServer layers

Browse the full notebook manifest, construct the correct WFS or WCS URL for any named tehsil, and load one vector layer without editing a URL.

Run each cell with **Shift+Enter**. The location controls default to the active KYL tehsil when this notebook is downloaded from CoRE Stack.

In [ ]:
import json, re, sys
from urllib.parse import urlencode
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
import geolibre

GEOSERVER_BASE = "https://geoserver.core-stack.org:8443/geoserver/"
SCOPE = json.loads("{\"state\":\"Jharkhand\",\"district\":\"Dumka\",\"tehsil\":\"Masalia\",\"bounds\":[86.89,23.94,87.24,24.28]}")
LAYER_SPECS = json.loads("[{\"id\":\"administrative_boundaries\",\"label\":\"Administrative Boundaries\",\"domain\":\"Demographic\",\"service\":\"WFS\",\"workspace\":\"panchayat_boundaries\",\"layerNameTemplate\":\"{district}_{tehsil}\",\"period\":\"Current published boundary\",\"description\":\"Village and panchayat boundaries for the selected tehsil.\"},{\"id\":\"demographics\",\"label\":\"Socio-Economic Profile\",\"domain\":\"Demographic\",\"service\":\"WFS\",\"workspace\":\"panchayat_boundaries\",\"layerNameTemplate\":\"{district}_{tehsil}\",\"period\":\"Census-derived profile\",\"description\":\"Population, households, social groups, and literacy attributes.\"},{\"id\":\"facilities\",\"label\":\"Facilities and Services Access\",\"domain\":\"Village\",\"service\":\"WFS\",\"workspace\":\"facilities_proximity\",\"layerNameTemplate\":\"facilities_{district}_{tehsil}\",\"period\":\"Current published analysis\",\"description\":\"Village-level distance and access indicators for essential services.\"},{\"id\":\"antyodaya\",\"label\":\"Mission Antyodaya Village Indicators (2020)\",\"domain\":\"Village\",\"service\":\"WFS\",\"workspace\":\"antyodaya_2020\",\"layerNameTemplate\":\"antyodaya20_{district}_{tehsil}\",\"period\":\"2020\",\"description\":\"Multi-domain village development indicators.\"},{\"id\":\"livestock\",\"label\":\"Village Livestock Census\",\"domain\":\"Village\",\"service\":\"WFS\",\"workspace\":\"livestocks\",\"layerNameTemplate\":\"livestocks_{district}_{tehsil}\",\"period\":\"Latest published census\",\"description\":\"Village livestock counts by species and sex.\"},{\"id\":\"mws_layers\",\"label\":\"Micro-watersheds and Hydrological Variables\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"mws_layers\",\"layerNameTemplate\":\"deltaG_well_depth_{district}_{tehsil}\",\"period\":\"2017-2018 to 2024-2025\",\"description\":\"Annual groundwater-storage change and MWS identifiers.\"},{\"id\":\"hydrological_boundaries\",\"label\":\"Hydrological Boundaries\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"mws_layers\",\"layerNameTemplate\":\"deltaG_well_depth_{district}_{tehsil}\",\"period\":\"Current MWS boundary\",\"description\":\"The same MWS geometry with a boundary-focused presentation.\"},{\"id\":\"mws_layers_fortnight\",\"label\":\"Fortnightly Hydrological Variables\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"mws_layers\",\"layerNameTemplate\":\"deltaG_fortnight_{district}_{tehsil}\",\"period\":\"July 2017 to June 2025\",\"description\":\"Fortnightly precipitation, evapotranspiration, runoff, and related water-balance values.\"},{\"id\":\"terrain_vector\",\"label\":\"Terrain Vector\",\"domain\":\"Land\",\"service\":\"WFS\",\"workspace\":\"terrain\",\"layerNameTemplate\":\"{district}_{tehsil}_cluster\",\"period\":\"Current terrain analysis\",\"description\":\"MWS-level plains, slopes, valleys, ridges, hills, and terrain cluster.\"},{\"id\":\"drainage\",\"label\":\"Drainage\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"drainage\",\"layerNameTemplate\":\"{district}_{tehsil}\",\"period\":\"Current published network\",\"description\":\"Drainage lines and stream order.\"},{\"id\":\"river\",\"label\":\"Rivers\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"river\",\"layerNameTemplate\":\"{district}_{tehsil}_river_vector\",\"period\":\"Current published network\",\"description\":\"River lines and available identifiers.\"},{\"id\":\"canal\",\"label\":\"Canals\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"canal\",\"layerNameTemplate\":\"{district}_{tehsil}_canal_vector\",\"period\":\"Current published network\",\"description\":\"Canal lines, project, purpose, and status where available.\"},{\"id\":\"remote_sensed_waterbodies\",\"label\":\"Remote-Sensed Waterbodies\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"swb\",\"layerNameTemplate\":\"surface_waterbodies_{district}_{tehsil}\",\"period\":\"2017-2018 to 2024-2025\",\"description\":\"Waterbody extent, seasonal area, use, ownership, storage, and beneficiaries.\"},{\"id\":\"soge\",\"label\":\"Stage of Groundwater Extraction\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"soge\",\"layerNameTemplate\":\"soge_vector_{district}_{tehsil}\",\"period\":\"Latest published assessment\",\"description\":\"Groundwater extraction, recharge, availability, and assessment class.\"},{\"id\":\"aquifer\",\"label\":\"Aquifer\",\"domain\":\"Hydrology\",\"service\":\"WFS\",\"workspace\":\"aquifer\",\"layerNameTemplate\":\"aquifer_vector_{district}_{tehsil}\",\"period\":\"Latest published assessment\",\"description\":\"Aquifer type, lithology, yield, depth, and management guidance.\"},{\"id\":\"cropping_intensity\",\"label\":\"Cropping Intensity\",\"domain\":\"Agriculture\",\"service\":\"WFS\",\"workspace\":\"crop_intensity\",\"layerNameTemplate\":\"{district}_{tehsil}_intensity\",\"period\":\"2017 to 2024\",\"description\":\"Annual cropping intensity and single-, double-, and triple-cropped area.\"},{\"id\":\"drought\",\"label\":\"Drought\",\"domain\":\"Agriculture\",\"service\":\"WFS\",\"workspace\":\"drought\",\"layerNameTemplate\":\"{district}_{tehsil}_drought\",\"period\":\"2017 to 2024\",\"description\":\"Dry spells and weekly mild, moderate, and severe drought indicators.\"},{\"id\":\"nrega\",\"label\":\"NREGA Assets\",\"domain\":\"NREGA\",\"service\":\"WFS\",\"workspace\":\"nrega_assets\",\"layerNameTemplate\":\"{district}_{tehsil}\",\"period\":\"Current published assets\",\"description\":\"NREGA asset locations, work categories, and expenditure fields.\"},{\"id\":\"green_credit\",\"label\":\"Green Credit Projects\",\"domain\":\"Restoration\",\"service\":\"WFS\",\"workspace\":\"green_credit\",\"layerNameTemplate\":\"{district}_{tehsil}_green_credit\",\"period\":\"Current published projects\",\"description\":\"Green Credit project polygons and available land information.\"},{\"id\":\"land_conflicts\",\"label\":\"Land Conflicts\",\"domain\":\"Industry\",\"service\":\"WFS\",\"workspace\":\"lcw\",\"layerNameTemplate\":\"{district}_{tehsil}_lcw_conflict\",\"period\":\"Current published records\",\"description\":\"Land-conflict locations with titles, dates, and source links.\"},{\"id\":\"industry\",\"label\":\"Industries and CSR\",\"domain\":\"Industry\",\"service\":\"WFS\",\"workspace\":\"factory_csr\",\"layerNameTemplate\":\"{district}_{tehsil}_factory_csr\",\"period\":\"Current published records\",\"description\":\"Industry and CSR locations with company classifications.\"},{\"id\":\"mining\",\"label\":\"Mining Sites\",\"domain\":\"Industry\",\"service\":\"WFS\",\"workspace\":\"mining\",\"layerNameTemplate\":\"{district}_{tehsil}_mining\",\"period\":\"Current published records\",\"description\":\"Published mining locations where available.\"},{\"id\":\"terrain\",\"label\":\"Terrain\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"terrain\",\"layerNameTemplate\":\"{district}_{tehsil}_terrain_raster\",\"period\":\"Current terrain analysis\",\"description\":\"Raster landform classes.\"},{\"id\":\"dem\",\"label\":\"Digital Elevation Model\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"dem\",\"layerNameTemplate\":\"{district}_{tehsil}_dem_raster\",\"period\":\"Current published DEM\",\"description\":\"Elevation raster for terrain and relief analysis.\"},{\"id\":\"clart\",\"label\":\"CLART\",\"domain\":\"Hydrology\",\"service\":\"WCS\",\"workspace\":\"clart\",\"layerNameTemplate\":\"{district}_{tehsil}_clart\",\"period\":\"Current published analysis\",\"description\":\"Recharge and water-harvesting intervention classes.\"},{\"id\":\"afforestation\",\"label\":\"Change Detection: Afforestation\",\"domain\":\"Restoration\",\"service\":\"WCS\",\"workspace\":\"change_detection\",\"layerNameTemplate\":\"change_{district}_{tehsil}_Afforestation\",\"period\":\"Published change period\",\"description\":\"Afforestation change classes.\"},{\"id\":\"deforestation\",\"label\":\"Change Detection: Deforestation\",\"domain\":\"Restoration\",\"service\":\"WCS\",\"workspace\":\"change_detection\",\"layerNameTemplate\":\"change_{district}_{tehsil}_Deforestation\",\"period\":\"Published change period\",\"description\":\"Deforestation change classes.\"},{\"id\":\"degradation\",\"label\":\"Change Detection: Degradation\",\"domain\":\"Restoration\",\"service\":\"WCS\",\"workspace\":\"change_detection\",\"layerNameTemplate\":\"change_{district}_{tehsil}_Degradation\",\"period\":\"Published change period\",\"description\":\"Land-degradation change classes.\"},{\"id\":\"urbanization\",\"label\":\"Change Detection: Urbanization\",\"domain\":\"Restoration\",\"service\":\"WCS\",\"workspace\":\"change_detection\",\"layerNameTemplate\":\"change_{district}_{tehsil}_Urbanization\",\"period\":\"Published change period\",\"description\":\"Urbanization change classes.\"},{\"id\":\"cropintensity\",\"label\":\"Change Detection: Crop Intensity\",\"domain\":\"Restoration\",\"service\":\"WCS\",\"workspace\":\"change_detection\",\"layerNameTemplate\":\"change_{district}_{tehsil}_CropIntensity\",\"period\":\"Published change period\",\"description\":\"Cropping-intensity change classes.\"},{\"id\":\"restoration\",\"label\":\"Restoration Opportunities\",\"domain\":\"Restoration\",\"service\":\"WCS\",\"workspace\":\"restoration\",\"layerNameTemplate\":\"restoration_{district}_{tehsil}_raster\",\"period\":\"Current published analysis\",\"description\":\"Mosaic, wide-scale restoration, and protection opportunities.\"},{\"id\":\"lulc_level_1_17_18\",\"label\":\"LULC Level 1 · 17-18\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_17_18_{district}_{tehsil}_level_3\",\"period\":\"2017-2018\",\"description\":\"Broad land-cover classes\"},{\"id\":\"lulc_level_1_18_19\",\"label\":\"LULC Level 1 · 18-19\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_18_19_{district}_{tehsil}_level_3\",\"period\":\"2018-2019\",\"description\":\"Broad land-cover classes\"},{\"id\":\"lulc_level_1_19_20\",\"label\":\"LULC Level 1 · 19-20\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_19_20_{district}_{tehsil}_level_3\",\"period\":\"2019-2020\",\"description\":\"Broad land-cover classes\"},{\"id\":\"lulc_level_1_20_21\",\"label\":\"LULC Level 1 · 20-21\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_20_21_{district}_{tehsil}_level_3\",\"period\":\"2020-2021\",\"description\":\"Broad land-cover classes\"},{\"id\":\"lulc_level_1_21_22\",\"label\":\"LULC Level 1 · 21-22\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_21_22_{district}_{tehsil}_level_3\",\"period\":\"2021-2022\",\"description\":\"Broad land-cover classes\"},{\"id\":\"lulc_level_1_22_23\",\"label\":\"LULC Level 1 · 22-23\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_22_23_{district}_{tehsil}_level_3\",\"period\":\"2022-2023\",\"description\":\"Broad land-cover classes\"},{\"id\":\"lulc_level_1_23_24\",\"label\":\"LULC Level 1 · 23-24\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_23_24_{district}_{tehsil}_level_3\",\"period\":\"2023-2024\",\"description\":\"Broad land-cover classes\"},{\"id\":\"lulc_level_1_24_25\",\"label\":\"LULC Level 1 · 24-25\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_24_25_{district}_{tehsil}_level_3\",\"period\":\"2024-2025\",\"description\":\"Broad land-cover classes\"},{\"id\":\"lulc_level_2_17_18\",\"label\":\"LULC Level 2 · 17-18\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_17_18_{district}_{tehsil}_level_3\",\"period\":\"2017-2018\",\"description\":\"Intermediate land-cover classes\"},{\"id\":\"lulc_level_2_18_19\",\"label\":\"LULC Level 2 · 18-19\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_18_19_{district}_{tehsil}_level_3\",\"period\":\"2018-2019\",\"description\":\"Intermediate land-cover classes\"},{\"id\":\"lulc_level_2_19_20\",\"label\":\"LULC Level 2 · 19-20\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_19_20_{district}_{tehsil}_level_3\",\"period\":\"2019-2020\",\"description\":\"Intermediate land-cover classes\"},{\"id\":\"lulc_level_2_20_21\",\"label\":\"LULC Level 2 · 20-21\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_20_21_{district}_{tehsil}_level_3\",\"period\":\"2020-2021\",\"description\":\"Intermediate land-cover classes\"},{\"id\":\"lulc_level_2_21_22\",\"label\":\"LULC Level 2 · 21-22\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_21_22_{district}_{tehsil}_level_3\",\"period\":\"2021-2022\",\"description\":\"Intermediate land-cover classes\"},{\"id\":\"lulc_level_2_22_23\",\"label\":\"LULC Level 2 · 22-23\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_22_23_{district}_{tehsil}_level_3\",\"period\":\"2022-2023\",\"description\":\"Intermediate land-cover classes\"},{\"id\":\"lulc_level_2_23_24\",\"label\":\"LULC Level 2 · 23-24\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_23_24_{district}_{tehsil}_level_3\",\"period\":\"2023-2024\",\"description\":\"Intermediate land-cover classes\"},{\"id\":\"lulc_level_2_24_25\",\"label\":\"LULC Level 2 · 24-25\",\"domain\":\"Land\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_24_25_{district}_{tehsil}_level_3\",\"period\":\"2024-2025\",\"description\":\"Intermediate land-cover classes\"},{\"id\":\"lulc_level_3_17_18\",\"label\":\"LULC Level 3 · 17-18\",\"domain\":\"Agriculture\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_17_18_{district}_{tehsil}_level_3\",\"period\":\"2017-2018\",\"description\":\"Detailed land-cover classes\"},{\"id\":\"lulc_level_3_18_19\",\"label\":\"LULC Level 3 · 18-19\",\"domain\":\"Agriculture\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_18_19_{district}_{tehsil}_level_3\",\"period\":\"2018-2019\",\"description\":\"Detailed land-cover classes\"},{\"id\":\"lulc_level_3_19_20\",\"label\":\"LULC Level 3 · 19-20\",\"domain\":\"Agriculture\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_19_20_{district}_{tehsil}_level_3\",\"period\":\"2019-2020\",\"description\":\"Detailed land-cover classes\"},{\"id\":\"lulc_level_3_20_21\",\"label\":\"LULC Level 3 · 20-21\",\"domain\":\"Agriculture\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_20_21_{district}_{tehsil}_level_3\",\"period\":\"2020-2021\",\"description\":\"Detailed land-cover classes\"},{\"id\":\"lulc_level_3_21_22\",\"label\":\"LULC Level 3 · 21-22\",\"domain\":\"Agriculture\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_21_22_{district}_{tehsil}_level_3\",\"period\":\"2021-2022\",\"description\":\"Detailed land-cover classes\"},{\"id\":\"lulc_level_3_22_23\",\"label\":\"LULC Level 3 · 22-23\",\"domain\":\"Agriculture\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_22_23_{district}_{tehsil}_level_3\",\"period\":\"2022-2023\",\"description\":\"Detailed land-cover classes\"},{\"id\":\"lulc_level_3_23_24\",\"label\":\"LULC Level 3 · 23-24\",\"domain\":\"Agriculture\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_23_24_{district}_{tehsil}_level_3\",\"period\":\"2023-2024\",\"description\":\"Detailed land-cover classes\"},{\"id\":\"lulc_level_3_24_25\",\"label\":\"LULC Level 3 · 24-25\",\"domain\":\"Agriculture\",\"service\":\"WCS\",\"workspace\":\"LULC_level_3\",\"layerNameTemplate\":\"LULC_24_25_{district}_{tehsil}_level_3\",\"period\":\"2024-2025\",\"description\":\"Detailed land-cover classes\"}]")
m = geolibre.connect()
MAP_LAYERS = {}

def geoserver_name(value):
    value = re.sub(r"[()]", "", str(value or "").strip().lower())
    return re.sub(r"_+", "_", re.sub(r"\s+", "_", value)).strip("_")

state_input = widgets.Text(value=SCOPE["state"], description="State:", layout=widgets.Layout(width="98%"))
district_input = widgets.Text(value=SCOPE["district"], description="District:", layout=widgets.Layout(width="98%"))
tehsil_input = widgets.Text(value=SCOPE["tehsil"], description="Tehsil:", layout=widgets.Layout(width="98%"))
display(widgets.VBox([
    widgets.HTML("<b>Study location</b><br><small>Change a name here, then rerun the data cells. No Python editing is needed.</small>"),
    state_input, district_input, tehsil_input,
]))

def selected_scope():
    return {
        "state": state_input.value.strip(),
        "district": geoserver_name(district_input.value),
        "tehsil": geoserver_name(tehsil_input.value),
    }

def get_spec(layer_id):
    return next(layer for layer in LAYER_SPECS if layer["id"] == layer_id)

def layer_url(layer_id, cql_filter=None):
    scope = selected_scope()
    spec = get_spec(layer_id)
    layer_name = spec["layerNameTemplate"].format(**scope)
    qualified = f'{spec["workspace"]}:{layer_name}'
    if spec["service"] == "WFS":
        params = {"service": "WFS", "version": "1.0.0", "request": "GetFeature",
                  "typeName": qualified, "outputFormat": "application/json", "srsName": "EPSG:4326"}
        if cql_filter:
            params["CQL_FILTER"] = cql_filter
        return f'{GEOSERVER_BASE}{spec["workspace"]}/ows?{urlencode(params)}'
    params = {"service": "WCS", "version": "2.0.1", "request": "GetCoverage",
              "CoverageId": qualified, "format": "geotiff", "compression": "LZW"}
    return f'{GEOSERVER_BASE}{spec["workspace"]}/wcs?{urlencode(params)}'

async def fetch_json(url, label="GeoServer layer"):
    try:
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            response = await pyfetch(url)
            if not response.ok:
                raise RuntimeError(f"HTTP {response.status}")
            return await response.json()
        import urllib.request
        with urllib.request.urlopen(url, timeout=90) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception as error:
        scope = selected_scope()
        raise RuntimeError(
            f'{label} is not available for {scope["district"]}/{scope["tehsil"]}, or GeoServer could not be reached: {error}'
        ) from error

async def load_geojson(layer_id, cql_filter=None):
    spec = get_spec(layer_id)
    if spec["service"] != "WFS":
        raise ValueError(f'{spec["label"]} is a raster. Use its WCS URL instead of loading it as GeoJSON.')
    data = await fetch_json(layer_url(layer_id, cql_filter), spec["label"])
    if data.get("type") != "FeatureCollection":
        raise RuntimeError(f'{spec["label"]} did not return GeoJSON features.')
    return data

def to_frame(data):
    rows = [dict(feature.get("properties") or {}) for feature in data.get("features", [])]
    return pd.DataFrame(rows)

def uid_column(frame):
    for name in ("uid", "UID", "MWS_UID", "MWS UID"):
        if name in frame.columns:
            return name
    raise KeyError("This layer has no recognised MWS identifier column.")

def with_uid(frame):
    result = frame.copy()
    result["uid"] = result[uid_column(result)].astype(str)
    return result

def numeric(frame, columns):
    return frame.loc[:, columns].apply(pd.to_numeric, errors="coerce")

def json_component(value, component):
    try:
        value = json.loads(value) if isinstance(value, str) else value
        return float(value.get(component)) if isinstance(value, dict) and value.get(component) is not None else np.nan
    except (TypeError, ValueError, json.JSONDecodeError):
        return np.nan

def component_values(frame, columns, component):
    return frame.loc[:, columns].apply(
        lambda series: series.map(lambda value: json_component(value, component))
    )

def features_for_uids(data, uids):
    wanted = {str(uid) for uid in uids}
    names = ("uid", "UID", "MWS_UID", "MWS UID")
    features = []
    for feature in data.get("features", []):
        properties = feature.get("properties") or {}
        value = next((properties.get(name) for name in names if properties.get(name) is not None), None)
        if str(value) in wanted:
            features.append(feature)
    return {"type": "FeatureCollection", "features": features}

def geojson_bounds(data):
    points = []
    def visit(value):
        if isinstance(value, list) and len(value) >= 2 and all(isinstance(v, (int, float)) for v in value[:2]):
            points.append(value[:2])
        elif isinstance(value, list):
            for item in value:
                visit(item)
    for feature in data.get("features", []):
        visit((feature.get("geometry") or {}).get("coordinates", []))
    if not points:
        return None
    xs, ys = zip(*points)
    return [min(xs), min(ys), max(xs), max(ys)]

def show_on_map(key, data, name, **style):
    if not data.get("features"):
        print(f"No features to map for {name}.")
        return None
    previous_layer_id = MAP_LAYERS.get(key)
    if previous_layer_id:
        try:
            m.remove_layer(previous_layer_id)
        except Exception:
            pass
    MAP_LAYERS[key] = m.add_geojson(data, name=name, **style)
    bounds = geojson_bounds(data)
    if bounds:
        m.fit_bounds(bounds)
    return MAP_LAYERS[key]

def year_columns(frame, prefix="", pattern=r"^\d{4}_\d{4}$"):
    return sorted(column for column in frame.columns if column.startswith(prefix) and re.search(pattern, column))

print(f'Ready for {SCOPE["tehsil"]}, {SCOPE["district"]}.')

In [ ]:
def manifest_table():
    columns = ["id", "label", "domain", "service", "workspace", "layerNameTemplate", "period", "description"]
    return pd.DataFrame(LAYER_SPECS)[columns].rename(columns={
        "id": "Layer ID", "label": "Dataset", "domain": "Theme", "service": "Download service",
        "workspace": "GeoServer workspace", "layerNameTemplate": "Layer-name template",
        "period": "Published period", "description": "What it contains",
    })

def describe_layer(layer_id):
    spec = get_spec(layer_id)
    return pd.DataFrame({"Field": ["Dataset", "Theme", "Service", "Period", "Description", "Generated URL"],
                         "Value": [spec["label"], spec["domain"], spec["service"], spec["period"],
                                   spec["description"], layer_url(layer_id)]})

## 1. Browse the complete manifest

WFS entries return vector geometry and attributes. WCS entries return analytical raster pixels; WMS map tiles are intentionally not used as data downloads.

In [ ]:
catalogue = manifest_table()
print(f"{len(catalogue)} GeoLibre layer presentations are available in this manifest.")
display(catalogue)

## 2. Choose one dataset

Change the location controls above or choose another dataset here, then rerun the next cell.

In [ ]:
layer_picker = widgets.Dropdown(
    options=[(layer["label"], layer["id"]) for layer in LAYER_SPECS],
    value="mws_layers", description="Dataset:", layout=widgets.Layout(width="98%"),
)
display(layer_picker)

## 3. Generate the download URL

Raster WCS coverages can be large, so this notebook prints their URL rather than downloading them automatically.

In [ ]:
layer_id = layer_picker.value
display(describe_layer(layer_id))
if get_spec(layer_id)["service"] == "WFS":
    chosen_geojson = await load_geojson(layer_id)
    chosen_table = to_frame(chosen_geojson)
    print(f"Loaded {len(chosen_table):,} features and {len(chosen_table.columns):,} attribute fields.")
    display(chosen_table.head(10))
else:
    print("Copy the generated WCS URL when you are ready to download the GeoTIFF coverage.")

## 4. Add a chosen vector to the map

Run this only after loading a WFS vector above.

In [ ]:
if get_spec(layer_id)["service"] == "WFS":
    show_on_map("manifest-choice", chosen_geojson, f'Notebook · {get_spec(layer_id)["label"]}',
                fillColor="#a78bfa", strokeColor="#4c1d95", fillOpacity=0.42)
else:
    print("Raster analysis uses the WCS GeoTIFF URL shown above; the existing GeoLibre project already provides its styled WMS map layer.")

## Interpretation

A missing layer or empty response means the source is not published for that named scope, not that the real-world phenomenon is absent. Preserve the dataset period and units when using downloaded attributes.